# Lab 04 — Runtime Configuration

This notebook is intentionally **DDL-free**.

It:
- defines runtime widgets and validates their values;
- builds reusable paths and table names;
- loads the version-controlled YAML data contracts;
- exposes one runtime-selected contract plus the v1/v2 references required by the schema-governance demonstrations.

It does **not** create catalogs, schemas, volumes, folders, or tables.

Run `lab04_00_setup` manually once when the Lab 4 structure must be created or verified. Production Job tasks may `%run ./lab04_00_config` safely.


In [0]:
def ensure_text_widget(name, default, label):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(name, default, choices, label):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("catalog", "dbr_dev", "01 Catalog")
ensure_text_widget("schema", "parvinbadalov", "02 Schema")
ensure_text_widget("volume_name", "lab04_silver_quality", "03 Volume")
ensure_dropdown_widget("volume_type", "external", ["external", "managed"], "04 Volume type")
ensure_text_widget("storage_account", "dlspl21databricks", "05 Storage account")
ensure_text_widget("container", "parvinbadalov", "06 ADLS container")
ensure_text_widget("external_volume_dir", "lab04_silver_quality", "07 External volume directory")
ensure_text_widget("source_file_name", "Online Retail.xlsx", "08 Source workbook")
ensure_text_widget("batch_id", "initial", "09 Batch ID")
ensure_dropdown_widget("contract_version", "v1", ["v1", "v2"], "10 Contract version")
ensure_dropdown_widget("trigger_type", "availableNow", ["availableNow", "once"], "11 Trigger type")
ensure_text_widget("max_files_per_trigger", "50", "12 Maximum files per trigger")
ensure_dropdown_widget("schema_policy", "fail", ["fail", "rescue", "evolve"], "13 Schema policy")
ensure_dropdown_widget("reset_demo_objects", "false", ["false", "true"], "14 Reset demo objects")
ensure_dropdown_widget("run_validation", "true", ["true", "false"], "15 Run validation")


In [0]:
import re

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume_name = dbutils.widgets.get("volume_name").strip()
volume_type = dbutils.widgets.get("volume_type").strip().lower()
storage_account = dbutils.widgets.get("storage_account").strip().lower()
container = dbutils.widgets.get("container").strip().lower()
external_volume_dir = dbutils.widgets.get("external_volume_dir").strip().strip("/")
source_file_name = dbutils.widgets.get("source_file_name").strip()
batch_id = dbutils.widgets.get("batch_id").strip()
contract_version = dbutils.widgets.get("contract_version").strip().lower()
trigger_type = dbutils.widgets.get("trigger_type").strip()
max_files_per_trigger = int(dbutils.widgets.get("max_files_per_trigger"))
schema_policy = dbutils.widgets.get("schema_policy").strip().lower()
reset_demo_objects = dbutils.widgets.get("reset_demo_objects").strip().lower() == "true"
run_validation = dbutils.widgets.get("run_validation").strip().lower() == "true"

identifier_pattern = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
for label, value in {
    "catalog": catalog,
    "schema": schema,
    "volume_name": volume_name,
}.items():
    if not identifier_pattern.fullmatch(value):
        raise ValueError(f"Invalid {label}: {value!r}")

storage_name_pattern = re.compile(r"^[a-z0-9-]+$")
for label, value in {
    "storage_account": storage_account,
    "container": container,
}.items():
    if not storage_name_pattern.fullmatch(value):
        raise ValueError(f"Invalid {label}: {value!r}")

if volume_type not in {"external", "managed"}:
    raise ValueError("volume_type must be 'external' or 'managed'")
if contract_version not in {"v1", "v2"}:
    raise ValueError("contract_version must be 'v1' or 'v2'")
if trigger_type not in {"availableNow", "once"}:
    raise ValueError("trigger_type must be 'availableNow' or 'once'")
if schema_policy not in {"fail", "rescue", "evolve"}:
    raise ValueError("schema_policy must be 'fail', 'rescue', or 'evolve'")
if max_files_per_trigger <= 0:
    raise ValueError("max_files_per_trigger must be greater than zero")


In [0]:
external_volume_url = (
    f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
    f"{external_volume_dir}"
)

volume_root = f"/Volumes/{catalog}/{schema}/{volume_name}"

paths = {
    "source": f"{volume_root}/source",
    "staging_initial": f"{volume_root}/staging/initial",
    "staging_incremental": f"{volume_root}/staging/incremental",
    "staging_evolved": f"{volume_root}/staging/evolved",
    "staging_invalid": f"{volume_root}/staging/invalid",
    "landing": f"{volume_root}/landing",
    "schema_bronze": f"{volume_root}/system/schema/bronze",
    "checkpoint_bronze": f"{volume_root}/system/checkpoints/bronze",
    "checkpoint_silver": f"{volume_root}/system/checkpoints/silver",
    "quarantine": f"{volume_root}/quarantine",
    "schema_mismatch": f"{volume_root}/test_data/schema_mismatch",
    "schema_evolution": f"{volume_root}/test_data/schema_evolution",
    "scd_changes": f"{volume_root}/test_data/scd_changes",
}

source_path = paths["source"]
landing_path = paths["landing"]
bronze_schema_path = paths["schema_bronze"]
bronze_checkpoint_path = paths["checkpoint_bronze"]
silver_checkpoint_path = paths["checkpoint_silver"]
source_file_path = f"{source_path}/{source_file_name}"


In [0]:
table_names = {
    "bronze": f"{catalog}.{schema}.lab04_bronze_retail",
    "silver_transactions": f"{catalog}.{schema}.lab04_silver_transactions",
    "quarantine": f"{catalog}.{schema}.lab04_quarantine",
    "quality_metrics": f"{catalog}.{schema}.lab04_quality_metrics",
    "product_scd0": f"{catalog}.{schema}.lab04_product_scd0",
    "product_scd1": f"{catalog}.{schema}.lab04_product_scd1",
    "product_scd2": f"{catalog}.{schema}.lab04_product_scd2",
    "product_scd3": f"{catalog}.{schema}.lab04_product_scd3",
    "product_scd4_current": f"{catalog}.{schema}.lab04_product_current",
    "product_scd4_history": f"{catalog}.{schema}.lab04_product_history",
    "product_scd6": f"{catalog}.{schema}.lab04_product_scd6",
    "schema_demo": f"{catalog}.{schema}.lab04_schema_demo",
    "column_mapping_demo": f"{catalog}.{schema}.lab04_column_mapping_demo",
}

bronze_table = table_names["bronze"]
silver_table = table_names["silver_transactions"]
quarantine_table = table_names["quarantine"]
quality_metrics_table = table_names["quality_metrics"]

expected_source_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
]


In [0]:
from pathlib import Path

try:
    import yaml
except ImportError as exc:
    raise ImportError(
        "PyYAML is required to load Lab 4 contracts. "
        "Install it in the Databricks environment/dependency configuration "
        "instead of running %pip from every pipeline notebook."
    ) from exc

contract_dir = Path(
    "/Workspace/Users/parvinbadalov@yahoo.com/"
    "Databricks-Academy-Lakehouse/"
    "labs/lab_04_silver_quality/contracts"
)

contract_files = {
    "v1": contract_dir / "online_retail_v1.yml",
    "v2": contract_dir / "online_retail_v2.yml",
}


def load_contract(version: str):
    version = version.strip().lower()

    if version not in contract_files:
        raise ValueError(
            f"Unknown contract version: {version!r}. "
            f"Expected one of {sorted(contract_files)}."
        )

    contract_path = contract_files[version]

    if not contract_path.exists():
        raise FileNotFoundError(f"Contract file was not found: {contract_path}")

    with open(contract_path, "r", encoding="utf-8") as file:
        contract = yaml.safe_load(file)

    if (
        not isinstance(contract, dict)
        or "contract" not in contract
        or "schema" not in contract
    ):
        raise ValueError(
            f"Invalid contract structure in {contract_path}. "
            "Expected top-level 'contract' and 'schema' sections."
        )

    return contract


# Both reference versions remain available for notebooks 08 and 09.
contract_v1 = load_contract("v1")
contract_v2 = load_contract("v2")

# Exactly one contract is selected for the current runtime.
active_contract = load_contract(contract_version)

yaml_version = f"v{active_contract['contract']['version']}"
if yaml_version != contract_version:
    raise AssertionError(
        f"Runtime selected {contract_version}, "
        f"but the loaded YAML declares {yaml_version}."
    )

active_contract_name = active_contract["contract"]["name"]
active_contract_status = active_contract["contract"].get("status", "unknown")
active_contract_column_count = len(active_contract["schema"]["columns"])


In [0]:
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

selected_contract_schema = StructType([
    StructField("runtime_selection", StringType(), False),
    StructField("contract_name", StringType(), False),
    StructField("yaml_version", StringType(), False),
    StructField("governance_status", StringType(), False),
    StructField("column_count", IntegerType(), False),
    StructField("supersedes", StringType(), True),
    StructField("runtime_selected", BooleanType(), False),
])

selected_contract_df = spark.createDataFrame(
    [
        (
            contract_version,
            active_contract_name,
            str(active_contract["contract"]["version"]),
            active_contract_status,
            int(active_contract_column_count),
            (
                str(active_contract["contract"]["supersedes"])
                if active_contract["contract"].get("supersedes") is not None
                else None
            ),
            True,
        )
    ],
    schema=selected_contract_schema,
)

if run_validation:
    display(selected_contract_df)

print(f"Runtime configuration ready: {catalog}.{schema}")
print(f"Volume: {volume_root}")
print(f"Source workbook: {source_file_path}")
print(
    f"Runtime contract: {active_contract_name} {contract_version} "
    f"(governance status: {active_contract_status})"
)
print(f"Schema policy: {schema_policy}")
